# Dynamic compaction

`CompactionSettings` controls a closed-loop series of transactional cell
changes. Each trial changes the cell, relaxes, checks guards, and is
accepted or rolled back before the next increment.

In [ ]:
# Compaction is configured independently from the recipe using it.
import tangle

## Every `CompactionSettings` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `target_type` | Stopping observable. | `volume_fraction`, `cell_volume`, `cell_lengths`, `mean_pressure`, `directional_pressure`, `penalty_energy` |
| `target_value` | Scalar target for scalar target types. | target-dependent |
| `target_values` | Three-component target for vector target types. | target-dependent xyz |
| `path` | Rule for distributing cell motion among axes. | `axis_weights`, `equal_pressure`, `stress_ratio`, `minimum_incremental_work` |
| `axis_weights` | Prescribed relative shortening for the axis-weight path. | nonnegative xyz |
| `active_axes` | Axes available to feedback-controlled paths. | three booleans |
| `stress_ratio` | Desired directional pressure ratio. | nonnegative xyz |
| `pressure_floor` | Numerical floor in pressure-ratio calculations. | pressure |
| `kinematics` | How geometry follows cell changes. | `rigid_fiber_centers`, `moving_walls`, `affine_vertices` |
| `cell_anchor` | Stationary fractional point while each cell axis changes. | xyz in [0,1] |
| `balance_opposing_faces` | Balances work between low and high faces. | boolean |
| `face_pressure_floor` | Floor used by opposing-face balancing. | pressure |
| `face_balance_strength` | Strength of opposing-face feedback. | 0–1 |
| `initial_log_strain` | First attempted logarithmic strain increment. | positive strain |
| `minimum_log_strain` | Smallest retry increment. | positive strain |
| `maximum_log_strain` | Largest grown increment. | positive strain |
| `growth_factor` | Increment multiplier after easy accepted steps. | greater than 1 |
| `shrink_factor` | Increment multiplier after rejected steps. | between 0 and 1 |
| `relax_iterations` | Relaxation work allotted to each increment window. | count |
| `maximum_shortening_over_minimum_diameter` | Caps an increment by the thinnest fiber size. | ratio |
| `maximum_penetration` | Rejects a trial exceeding this overlap. | m |
| `maximum_bend_ratio` | Rejects a trial exceeding this curvature utilization. | ratio |
| `maximum_pressure` | Pressure guard. | pressure |
| `maximum_penalty_energy` | Formation-energy guard. | energy |
| `maximum_steps` | Maximum accepted/retried compaction steps. | count |
| `maximum_relax_windows` | Maximum windows spent settling one trial. | count |
| `contact_energy_stiffness` | Contact contribution to the formation penalty. | model stiffness |
| `stretch_energy_stiffness` | Stretch contribution to the formation penalty. | model stiffness |
| `bending_energy_stiffness` | Bending contribution to the formation penalty. | model stiffness |
| `target_tolerance` | Relative/absolute acceptance tolerance for the target. | target-dependent |

In [ ]:
# Read defaults from the compiled extension instead of duplicating
# them in documentation that could become stale.
compaction = tangle.CompactionSettings()
fields = ['target_type', 'target_value', 'target_values', 'path', 'axis_weights', 'active_axes', 'stress_ratio', 'pressure_floor', 'kinematics', 'cell_anchor', 'balance_opposing_faces', 'face_pressure_floor', 'face_balance_strength', 'initial_log_strain', 'minimum_log_strain', 'maximum_log_strain', 'growth_factor', 'shrink_factor', 'relax_iterations', 'maximum_shortening_over_minimum_diameter', 'maximum_penetration', 'maximum_bend_ratio', 'maximum_pressure', 'maximum_penalty_energy', 'maximum_steps', 'maximum_relax_windows', 'contact_energy_stiffness', 'stretch_energy_stiffness', 'bending_energy_stiffness', 'target_tolerance']
{name: getattr(compaction, name) for name in fields}

## Common target/path combinations

`volume_fraction()` is the concise constructor. Other stopping targets
are selected by changing `target_type` and the scalar `target_value` or
vector `target_values`. The path is independent of the target.

In [ ]:
# axis_weights map to [x, y, z]; only the bounded z direction shortens.
compaction = tangle.CompactionSettings.volume_fraction(
    0.40, axis_weights=[0.0, 0.0, 1.0]
)
compaction.kinematics = "moving_walls"
compaction.cell_anchor = [0.5, 0.5, 0.5]  # both z faces move
compaction.balance_opposing_faces = True
compaction.maximum_penetration = 0.1e-6
compaction.maximum_bend_ratio = 1.05

# Copies make it easy to compare paths without rebuilding every guard.
equal_pressure = compaction.copy()
equal_pressure.path = "equal_pressure"
equal_pressure.active_axes = [True, True, True]

target_lengths = compaction.copy()
target_lengths.target_type = "cell_lengths"
target_lengths.target_values = [0.8e-3, 0.8e-3, 1.2e-3]

In [ ]:
# Stage overrides affect only this compaction operation's relaxation.
recipe = tangle.Recipe(tangle.Cell([1e-3, 1e-3, 2e-3]))
stage_overrides = tangle.RelaxationOverrides()
stage_overrides.contact_aggregation = "deepest_only"
recipe.compact(compaction, stage_overrides)
recipe.operations()